In [ ]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

DATA_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

def load_dataset() -> pd.DataFrame:
    """Download and prepare the labeled message dataset."""
    with urlopen(DATA_URL, timeout=30) as response:
        archive_bytes = response.read()
    with ZipFile(BytesIO(archive_bytes)) as archive:
        filename = next(name for name in archive.namelist() if name.endswith("SMSSpamCollection"))
        with archive.open(filename) as dataset_file:
            data = pd.read_csv(
                dataset_file,
                sep="\t",
                header=None,
                names=["label", "text"],
                encoding="utf-8",
            )

    data["label"] = data["label"].astype(str).str.strip().str.lower()
    data["text"] = data["text"].astype(str).str.strip()
    data = data[data["label"].isin(["ham", "spam"])].dropna()
    return data[data["text"] != ""].drop_duplicates(subset="text")

def main() -> None:
    messages = load_dataset()
    print("Downloaded messages:", len(messages))
    print("Class counts:\n", messages["label"].value_counts())
    x_train, x_test, y_train, y_test = train_test_split(
        messages["text"],
        messages["label"],
        test_size=0.2,
        random_state=42,
        stratify=messages["label"],
    )
    model = Pipeline(
        [
            ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
            ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ]
    )
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)
    print(f"\nTest accuracy: {accuracy_score(y_test, predictions):.3f}")
    print("\nClassification report:")
    print(classification_report(y_test, predictions, zero_division=0))
    print("\nTry a message (or type 'quit'):")
    while True:
        message = input("Message: ").strip()
        if message.lower() in {"quit", "exit", "q"}:
            break
        if message:
            label = model.predict([message])[0]
            print("Prediction:", "SPAM" if label == "spam" else "HAM (not spam)")

if __name__ == "__main__":
    main()